In [ ]:
import pandas as pd
import numpy as np
import sys 
import os 

from sklearn import set_config
from lifelines.statistics import logrank_test
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from typing import Tuple
from sklearn.model_selection import GroupKFold
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.plots import Plots
from src.utils.cox_models import Cox_regression, p_values_Cox_regression
%load_ext autoreload
%autoreload 2
%matplotlib inline


<font size="4">Preprocessing and Plots</font>

In [ ]:
pp = Preprocessor()
plots_class = Plots()

In [ ]:
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_data = pp.clean_columns_dataset(df_clinical_data)
list_df = pp.total_type_len_type_cancer(df_clinical_data)
df_clinical_data["Tumor-Cancer"] = list_df
df_clinical_data["Tumor-Cancer"].unique()

In [ ]:
df_mRNA_raw_data = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")


<font size="4">Merging the clinical DataSet and mRNA-Seq</font>

In [ ]:
df_merged = pp.merge_datasets(df_clinical_data, df_mRNA_raw_data)

In [ ]:
comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
    ["Tumor-Cancer"] + list(df_merged.columns[1:20441])
]
comparation_df["Tumor-Cancer"].unique()

In [ ]:
comparation_clean_df = pp.eliminate_zero_genes(comparation_df, "Tumor-Cancer")


In [ ]:
comparation_clean_df.iloc[:, 1:-1]

In [ ]:
print(f"Samples: {comparation_clean_df.shape[0]}, Genes: {comparation_clean_df.shape[1]}")

In [ ]:
comparation_clean_df.iloc[:, 1:-1] = np.log2(comparation_clean_df.iloc[:, 1:-1] + 1)


In [ ]:
comparation_clean_df.iloc[:, 1:-1]

#print(min(comparation_clean_df[1]))
print(max(comparation_clean_df[comparation_clean_df["Tumor-Cancer"] == "Luminal A"][0]))
print(min(comparation_clean_df[comparation_clean_df["Tumor-Cancer"] == "Luminal A"][0]))

In [ ]:
min_range = 1
max_range = 21

plots_class.box_plot(data=comparation_clean_df, title="Box plots Luminal A before Limma",
         type_cancer="Luminal A", range_min=min_range,range_max=max_range)

plots_class.box_plot(data=comparation_clean_df, title="Box plots Luminal B before Limma",
         type_cancer="Luminal B", range_min=min_range,range_max=max_range)

plots_class.box_plot(data=comparation_clean_df, title="Box plots TNBC before Limma",
         type_cancer="TNBC", range_min=min_range,range_max=max_range)

plots_class.box_plot(data=comparation_clean_df, title="Box plots HER2-enriched before Limma",
         type_cancer="HER2-enriched", range_min=min_range,range_max=max_range)


In [ ]:

plots_class.density_plot(data=comparation_clean_df, title="Density Plot Luminal A before Limma",
         type_cancer="Luminal A", range_min=min_range,range_max=max_range)

plots_class.density_plot(data=comparation_clean_df, title="Density Plot Luminal B before Limma",
         type_cancer="Luminal B", range_min=min_range,range_max=max_range)

plots_class.density_plot(data=comparation_clean_df, title="Density Plot TNBC before Limma",
         type_cancer="TNBC", range_min=min_range,range_max=max_range)

plots_class.density_plot(data=comparation_clean_df, title="Density Plot HER2-enriched before Limma",
         type_cancer="HER2-enriched", range_min=min_range,range_max=max_range)


In [ ]:
plots_class.PCA_4_scatter_matrix_log2(df=comparation_clean_df, cancer_types=["Luminal A", "Luminal B", "TNBC", "HER2-enriched"])

<font size="4">Cox Regression and Multi Cox Regression</font>

In [ ]:
def split_data_time_months(df_mRNA: pd.DataFrame, 
                         df_clinical: pd.DataFrame,
                         gene: str,
                         time : int) -> Tuple:
    
    df_single_gene = pp.gene_to_long(df_mRNA, gene)
    
    df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
    
    df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(
        df_gene_merged["Overall Survival (Months)"], errors="coerce"
    )
    
    status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
    df_gene_merged["event"] = status.str.contains("DECEASED", na=False)
    
    df_gene_merged["event_60"] = df_gene_merged["event"].copy()
    df_gene_merged["time_60"] = np.minimum(df_gene_merged["Overall Survival (Months)"], time)
    
    df_gene_merged.loc[
        df_gene_merged["Overall Survival (Months)"] > time, "event_60"
    ] = False
    
    df_gene_merged["expression"] = pd.to_numeric(
        df_gene_merged["expression"], errors="coerce"
    )
    
    df_gene_merged["expression"] = np.log2(df_gene_merged["expression"] +   1)
    
    df_gene_merged = df_gene_merged.dropna(subset=["time_60", "expression"]).copy()
    
    X = df_gene_merged[["expression"]]
    
    Y_surv = Surv.from_dataframe(
        event="event_60",
        time="time_60",
        data=df_gene_merged
    )
    
    return X, Y_surv, df_gene_merged

In [ ]:
clean_mRNA_df = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")

X_ESR1, Y_surv_ESR17, df_gene_merged_ESR1 = split_data_time_months(clean_mRNA_df,df_clinical_data, "ESR1", 60)
X_train_ESR1, X_test_ESR1, Y_train_ESR1, Y_test_ESR1 = train_test_split(
    X_ESR1, Y_surv_ESR17, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_ESR1, survival_curve_ESR1, risk_curve_ESR1 = Cox_regression(X_train=X_train_ESR1, 
                                                                                     Y_train=Y_train_ESR1, X_test=X_test_ESR1, draw_plot=True, title="ESR1")

In [ ]:
df_life_ESR1 = df_gene_merged_ESR1[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_ESR1["expression"] = pd.to_numeric(df_life_ESR1["expression"], errors="coerce")
df_life_ESR1["Overall Survival (Months)"] = pd.to_numeric(df_life_ESR1["Overall Survival (Months)"], errors="coerce")
df_life_ESR1 = df_life_ESR1.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_ESR1["time_60"] = np.minimum(df_life_ESR1["Overall Survival (Months)"], 60)
df_life_ESR1["event_60"] = df_life_ESR1["event"].copy()
df_life_ESR1.loc[df_life_ESR1["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_ESR1_cox = df_life_ESR1[["expression", "time_60", "event_60"]].copy()
p_value_ESR1 = p_values_Cox_regression(
    df_life_ESR1_cox,
    event_col="event_60",
    duration_col="time_60"
)

p_value_ESR1

In [ ]:
thr = df_gene_merged_ESR1["expression"].median()
low_group = df_gene_merged_ESR1[df_gene_merged_ESR1["expression"] < thr]
high_group = df_gene_merged_ESR1[df_gene_merged_ESR1["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

In [ ]:
clean_mRNA_df = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")

X_AURKA, Y_surv_AURKA, df_gene_merged_AURKA = split_data_time_months(clean_mRNA_df,df_clinical_data, "AURKA", 60)
X_train_AURKA, X_test_AURKA, Y_train_AURKA, Y_test_ESR1 = train_test_split(
    X_AURKA, Y_surv_AURKA, train_size=0.80, test_size=0.20, random_state=42
)
betas_AURKA, chp_predict_AURKA, survival_curve_AURKA, risk_curve_AURKA = Cox_regression(
    X_train=X_train_AURKA, Y_train=Y_train_AURKA, X_test=X_test_AURKA, draw_plot=True, title="AURKA"
)

In [ ]:
df_life_AURKA = df_gene_merged_AURKA[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_AURKA["expression"] = pd.to_numeric(df_life_AURKA["expression"], errors="coerce")
df_life_AURKA["Overall Survival (Months)"] = pd.to_numeric(df_life_AURKA["Overall Survival (Months)"], errors="coerce")
df_life_AURKA = df_life_AURKA.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_AURKA["time_60"] = np.minimum(df_life_AURKA["Overall Survival (Months)"], 60)
df_life_AURKA["event_60"] = df_life_AURKA["event"].copy()
df_life_AURKA.loc[df_life_AURKA["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_AURKA_cox = df_life_AURKA[["expression", "time_60", "event_60"]].copy()
p_value_AURKA = p_values_Cox_regression(
    df_life_AURKA_cox,
    event_col="event_60",
    duration_col="time_60"
)
#Outcome y endpoint
p_value_AURKA

In [ ]:
thr = df_gene_merged_AURKA["expression"].median()
low_group = df_gene_merged_AURKA[df_gene_merged_AURKA["expression"] < thr]
high_group = df_gene_merged_AURKA[df_gene_merged_AURKA["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

In [ ]:
clean_mRNA_df = pp.eliminate_zero_genes(df_mRNA_raw_data, "Hugo_Symbol")

X_MKI67, Y_surv_MKI67, df_gene_merged_MKI67 = split_data_time_months(clean_mRNA_df,df_clinical_data, "MKI67", 60)
X_train_MKI67, X_test_MKI67, Y_train_MKI67, Y_test_MKI67 = train_test_split(
    X_MKI67, Y_surv_MKI67, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_MKI67, survival_curve_MKI67, risk_curve_MKI67 = Cox_regression(
    X_train=X_train_MKI67, Y_train=Y_train_MKI67, X_test=X_test_MKI67, draw_plot=True, title="MKI67"
)

In [ ]:
df_life_MKI67 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]].copy()
df_life_MKI67["expression"] = pd.to_numeric(df_life_MKI67["expression"], errors="coerce")
df_life_MKI67["Overall Survival (Months)"] = pd.to_numeric(df_life_MKI67["Overall Survival (Months)"], errors="coerce")
df_life_MKI67 = df_life_MKI67.dropna(subset=["expression", "event", "Overall Survival (Months)"])
df_life_MKI67["time_60"] = np.minimum(df_life_MKI67["Overall Survival (Months)"], 60)
df_life_MKI67["event_60"] = df_life_MKI67["event"].copy()
df_life_MKI67.loc[df_life_MKI67["Overall Survival (Months)"] > 60, "event_60"] = False
df_life_MKI67_cox = df_life_MKI67[["expression", "time_60", "event_60"]].copy()
p_value_MKI67 = p_values_Cox_regression(
    df_life_MKI67_cox,
    event_col="event_60",
    duration_col="time_60"
)

p_value_MKI67

In [ ]:
thr = df_gene_merged_MKI67["expression"].median()
low_group = df_gene_merged_MKI67[df_gene_merged_MKI67["expression"] < thr]
high_group = df_gene_merged_MKI67[df_gene_merged_MKI67["expression"] >= thr]

results = logrank_test(
        durations_A=low_group["time_60"],
        durations_B=high_group["time_60"],
        event_observed_A=low_group["event_60"],
        event_observed_B=high_group["event_60"]
    )
print(results)

<font size="4">GroupKFold for all the genes</font>

In [ ]:
#GroupKFold
group_k_fold = GroupKFold(n_splits=10)
group_k_fold.get_n_splits()
df_merged = pp.merge_datasets(df_clinical_data, clean_mRNA_df)
expressions_genes_cols = df_merged.iloc[1:20441].sample(5, axis="columns")



In [ ]:
cols = ["Sample ID","Tumor-Cancer", "Overall Survival Status", "Overall Survival (Months)"] + list(expressions_genes_cols)

comparation_df = df_merged.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "TNBC", "HER2-enriched"]),
    cols
]

comparation_df = pp.eliminate_zero_genes(comparation_df, "Tumor-Cancer", threshold=0.8)


In [ ]:
status = comparation_df["Overall Survival Status"].astype(str).str.strip()

comparation_df["event"] = status.str.contains("DECEASED", na=False)

comparation_df = comparation_df.dropna(subset=["Overall Survival (Months)"])

groups = comparation_df["Sample ID"]
comparation_df = comparation_df.drop(columns="Sample ID")

X = comparation_df.iloc[:, 3:-1]

Y = Surv.from_dataframe(
    event="event",
    time="Overall Survival (Months)",
    data=comparation_df
)

for i, (train_index, test_index) in enumerate(group_k_fold.split(X, Y, groups=groups)):
    print(f"Fold {i}:")
    X_train = X.iloc[train_index]
    Y_train = Y[train_index]
    
    X_test = X.iloc[test_index]
    betas, chp_predict, survival_curve, risk_curve = Cox_regression(X_train,
                                                                    Y_train,
                                                                    X_test,
                                                                    draw_plot=False
                                                                    ) 
    
    
    #Lista de significativos , intento con todos, MultiVariante -> Le añado unos cuantos 
    if i > 2:
        break

In [ ]:
print(betas)
print(chp_predict)
print(survival_curve)
print(risk_curve)